# Worm Moebius

In this notebook, we implement the worm algorithm for the Moebius code for $d = 2 p$ with $p$ odd prime. 

In [1]:
from jax.typing import ArrayLike
import jax.numpy as jnp
from jax.sharding import Mesh, PartitionSpec, NamedSharding
from functools import partial
import numpy as np
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
from coherentinfo.worm import (
    run_worm
)
import os
import jax
n_cpus = os.cpu_count()
n_used_cpus = n_cpus
print("Number of CPUs available: {}".format(n_cpus))
print("Number of used CPUs: {}".format(n_used_cpus))
jax.config.update('jax_num_cpu_devices', n_used_cpus)
jax.config.update("jax_log_compiles", True)
# Devices assumed by JAX
print(jax.devices())

Number of CPUs available: 16
Number of used CPUs: 16
[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7), CpuDevice(id=8), CpuDevice(id=9), CpuDevice(id=10), CpuDevice(id=11), CpuDevice(id=12), CpuDevice(id=13), CpuDevice(id=14), CpuDevice(id=15)]


In [ ]:
length = 5  # 5
width = 5  # 5
p = 3
d = 2 * p
moebius_code = MoebiusCodeTwoOddPrime(length=length, width=width, d=d)
h_z = moebius_code.h_z
h_x = moebius_code.h_x
h_z_mod_2 = moebius_code.h_z_mod_2
h_z_mod_p = moebius_code.h_z_mod_p
h_x_mod_2 = moebius_code.h_x_mod_2
h_x_mod_p = moebius_code.h_x_mod_p
logical_x = moebius_code.logical_x
logical_z = moebius_code.logical_z
num_plaquette, num_edges = h_x.shape
gamma_t = 0.3
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=d, gamma_t=gamma_t
)

# error_master_key = jax.random.PRNGKey(42)
# initial_error = em_lindblad.generate_random_error(error_key)
# initial_error_mod_2 = jnp.mod(initial_error, 2)
# initial_error_mod_p = jnp.mod(initial_error, p)
# # Here we consider the full syndrome including the plaquette
# # we usually remove because of the constraint as this simplified the
# # coding of the worm algorithm. In fact, in this the syndromes will
# # always be annihilated in pairs, and the total number of syndromes is
# # always even as one can check numerically.
# syndrome = moebius_code.get_plaquette_syndrome(initial_error)
# syndrome_mod_2 = jnp.mod(syndrome, 2)
# syndrome_mod_p = jnp.mod(syndrome, p)
# num_plaquette, num_edges = h_x.shape

In [ ]:
num_samples = 10 * n_cpus
num_worms = 10 * n_cpus
master_worm_key = jax.random.PRNGKey(144)

worm_keys = jax.random.split(
    master_worm_key, num=num_samples * num_worms).reshape(num_samples, num_worms, 2)

error_master_key = jax.random.PRNGKey(42)
error_keys = jax.random.split(error_master_key, num_samples)


def generate_initial_worm_errors(
    key: ArrayLike,
    error_model: ErrorModelLindbladTwoOddPrime
) -> ArrayLike:
    initial_error = error_model.generate_random_error(key)
    initial_error_mod_2 = jnp.mod(initial_error, 2)
    initial_error_mod_p = jnp.mod(initial_error, p)
    initial_worm_error = jnp.vstack((initial_error_mod_2, initial_error_mod_p))
    return initial_worm_error


initial_worm_errors = jax.vmap(
    generate_initial_worm_errors, in_axes=(0, None))(error_keys, em_lindblad)

In [50]:
# Sharding the arrays
devices = jax.devices()  # Assuming this returns 16 devices
mesh = Mesh(devices, ('batch',))

sharding_for_keys = NamedSharding(mesh,  PartitionSpec('batch', None))
worm_keys_sharded = jax.device_put(worm_keys, sharding_for_keys)

# 2. Define sharding: Split the 0th axis across 'batch', leave others whole
sharding_for_error = NamedSharding(mesh,  PartitionSpec('batch', None, None))
initial_worm_errors_sharded = jax.device_put(
    initial_worm_errors, sharding_for_error)

In [51]:
jax.debug.visualize_array_sharding(initial_worm_errors_sharded[:, 1, :])

               CPU 0               
               CPU 1               
               CPU 2               
               CPU 3               
               CPU 4               
               CPU 5               
               CPU 6               
               CPU 7               
               CPU 8               
               CPU 9               
              CPU 10               
              CPU 11               
              CPU 12               
              CPU 13               
              CPU 14               
              CPU 15               

In [52]:
jax.debug.visualize_array_sharding(worm_keys_sharded[:, :, 0])

          CPU 0          
          CPU 1          
          CPU 2          
          CPU 3          
          CPU 4          
          CPU 5          
          CPU 6          
          CPU 7          
          CPU 8          
          CPU 9          
         CPU 10          
         CPU 11          
         CPU 12          
         CPU 13          
         CPU 14          
         CPU 15          

In [53]:
# master_worm_key = jax.random.PRNGKey(144)
# num_worms = 100
# worm_keys = jax.random.split(master_worm_key, num_worms)

# error_master_key = jax.random.PRNGKey(42)
# num_samples = 100
# worm_keys = jax.random.split(master_worm_key, num_worms)
# initial_error = em_lindblad.generate_random_error(error_key)

max_steps = 1000
initial_worm_state = {}
# worm_error = jnp.vstack(
#     (initial_error_mod_2, initial_error_mod_p))
initial_worm_state["worm_success"] = False
initial_worm_state["accepted_moves"] = 0
initial_worm_state["attempted_moves"] = 0

run_worm_partial = partial(
    run_worm,
    initial_worm_state=initial_worm_state,
    h_error_mod_p=h_z_mod_p,
    h_mod_p=h_x_mod_p,
    error_model=em_lindblad,
    num_stabs=num_plaquette,
    max_worm_steps=max_steps
)

# First over keys
run_worm_vmap = jax.vmap(run_worm_partial, in_axes=(None, 0))
# Then over initial errors
run_worm_vmap = jax.vmap(run_worm_vmap, in_axes=(0, 0))
run_worm_jit = jax.jit(run_worm_vmap)
new_worm_state = run_worm_jit(initial_worm_errors_sharded, worm_keys)

# index = 0
# print("Number of accepted moves: {}".format(new_worm_state["accepted_moves"][index]))
# print("Number of attempted moves: {}".format(
#     new_worm_state["attempted_moves"][index]))

In [54]:
index = 28
print("Number of accepted moves: {}".format(
    new_worm_state["accepted_moves"][index]))
print("Number of attempted moves: {}".format(
    new_worm_state["attempted_moves"][index]))
print("Success: {}".format(
    new_worm_state["worm_success"][index]))

Number of accepted moves: [ 4 16  2  2 14  2 71  2  2  2  2  2 78  2 26  2  8  6 40  4  2  2  2  2
  4 18  2 10  8  4  2  2  4  2  4 30  2  2 16  2  2  2  2 12 32  2  2  4
  4 20  8 34  4  4  4  2  2 22  2  4  2  2  2  2  2  2  2  2  2  6  6  2
  2  2  2 40  2  0  6  2  2  4 11  2 30  2 14 10  2  4  2  2  2  2  2 10
  8  2  2  2  2  2  2  2 35 12 45  2  2  2  2  2  2  2 22  2  6  2  4 38
 18 12  2  2  2 12  2  6 27  2  6  6  2  4  2  2  2  2  2  6  2  2  4  2
  2  2  2  2  8  2  2 14  4 44  2  2  2  6  6  2]
Number of attempted moves: [  16  161   24   12  182   46 1000   31   31   28   35    3 1000   13
  421   22  166   70  703  145  158   10   28   39   96  215    8  139
   45   43   47   25   61   23   30  440   52   28  568  427  572   71
   53  186  363  149   25   30   60  396  106  306   18   44  609   29
  347  244   61   25   60   96   87    8   24   63    3   98   10   39
  229    9   16   21   64  651   16 1000   30   25   73   23 1000   38
  234  128   98   99    6   75  1

In [55]:
print(new_worm_state["head"])
print(initial_worm_state["head"])

[[ 90  54  97 ...  56  18  10]
 [ 10  79  35 ...  84  36  11]
 [ 71  23  87 ...   0  98  40]
 ...
 [ 31 101  69 ...  21 109  18]
 [ 35  29 103 ...  29 113 101]
 [111  87  40 ...  75  57 102]]
VmapTracer<int32[]>


In [56]:
initial_worm_errors[0][0]

Array([0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1,
       0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1,
       1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0,
       0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1,
       0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1,
       0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0], dtype=int32)

In [59]:
index_a = 64
index_b = 1
print(jnp.max(
    jnp.abs(jnp.mod(new_worm_state["worm_error"][index_a, index_b][0, :] - initial_worm_errors[index_a][0, :], 2))))

1


In [60]:
print(jnp.max(
    jnp.abs(jnp.mod(new_worm_state["worm_error"][index_a, index_b][1, :] - initial_worm_errors[index_a][1, :], p))))

1


In [61]:
syndrome_mod_2 = jnp.mod(h_x_mod_2 @ initial_worm_errors[index_a][0, :], 2)
new_syndrome_mod_2 = jnp.mod(
    h_x_mod_2 @ new_worm_state["worm_error"][index_a, index_b][0, :], 2)

jnp.mod(new_syndrome_mod_2 - syndrome_mod_2, 2)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [62]:
syndrome_mod_p = jnp.mod(h_x_mod_p @ initial_worm_errors[index_a][1, :], p)
new_syndrome_mod_p = jnp.mod(
    h_x_mod_p @ new_worm_state["worm_error"][index_a, index_b][1, :], p)

jnp.mod(new_syndrome_mod_p - syndrome_mod_p, p)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)